# Drop Existing Constraints

In [0]:
# 1. Get FK dependencies on a table
deps = spark.sql("""
SELECT
  fk.constraint_catalog,
  fk.constraint_schema,
  fk.table_name,
  fk.constraint_name
FROM system.information_schema.table_constraints fk
JOIN system.information_schema.referential_constraints rc
  ON fk.constraint_name = rc.constraint_name
WHERE fk.constraint_type = 'FOREIGN KEY'
  AND rc.unique_constraint_name = 'counties_pk'
""").collect()
for r in deps:
    table_fqn = f"{r.constraint_catalog}.{r.constraint_schema}.{r.table_name}"
    
    print(f"Dropping {table_fqn}.{r.constraint_name}")
    spark.sql(f"""
        ALTER TABLE {table_fqn}
        DROP CONSTRAINT {r.constraint_name}
    """)

In [0]:
%sql
SELECT
  fk.constraint_catalog,
  fk.constraint_schema,
  fk.table_name,
  fk.constraint_name
FROM system.information_schema.table_constraints fk
JOIN system.information_schema.referential_constraints rc
  ON fk.constraint_name = rc.constraint_name
WHERE fk.constraint_type = 'FOREIGN KEY'
  AND rc.unique_constraint_name = 'counties_pk'

In [0]:
%sql
SELECT
  tc.constraint_catalog,
  tc.constraint_schema,
  tc.table_name,
  tc.constraint_name
FROM system.information_schema.table_constraints tc
WHERE tc.constraint_type = 'PRIMARY KEY'

In [0]:
pk_df = spark.sql("""
SELECT
  tc.constraint_catalog,
  tc.constraint_schema,
  tc.table_name,
  tc.constraint_name
FROM system.information_schema.table_constraints tc
WHERE tc.constraint_type = 'PRIMARY KEY'
""")

pk_rows = pk_df.collect()

for r in pk_rows:
    table_fqn = f"{r.constraint_catalog}.{r.constraint_schema}.{r.table_name}"
    try:
        spark.sql(f"""
            ALTER TABLE {table_fqn}
            DROP CONSTRAINT {r.constraint_name}
            CASCADE
        """)
    except Exception as e:
        print(table_fqn)
        print(r.constraint_name)
        print(e)

# Create Primary Key Constraints

In [0]:

primary_keys = [
    ['silver_dev.demographics.county_population', 'county_fips'],
    ['silver_dev.demographics.county_poverty', 'county_fips'],
    ['silver_dev.demographics.poverty_attributes', 'attribute_name'],
    ['silver_dev.demographics.ruca_census_tract', 'tract_fips_20'],
    ['silver_dev.demographics.ruca_codes', 'ruca_code'],
    ['silver_dev.geo.states', 'state_fips'],
    ['silver_dev.geo.counties', 'county_fips'],
    ['silver_dev.healthcare.hospitals', 'facility_id'],
    ['silver_dev.healthcare.providers', 'npi'],
    ['gold_dev.demographics.ruca_county', 'county_fips'],
    ['gold_dev.unified.county_unified', 'county_fips']
]

primary_keys = [
    ['silver_dev.demographics.county_population', 'county_fips'],
    ['silver_dev.demographics.county_poverty', 'county_fips'],
    ['silver_dev.demographics.poverty_attributes', 'attribute_name'],
    ['silver_dev.demographics.ruca_census_tract', 'tract_fips_20'],
    ['silver_dev.demographics.ruca_codes', 'ruca_code'],
    ['silver_dev.geo.states', 'state_fips'],
    ['silver_dev.geo.counties', 'county_fips'],
    ['silver_dev.healthcare.hospitals', 'facility_id'],
    ['silver_dev.healthcare.providers', 'npi'],
    ['gold_dev.demographics.ruca_county', 'county_fips'],
    ['gold_dev.unified.county_unified', 'county_fips']
]

for table_fqn, column in primary_keys:
    table_name = table_fqn.split('.')[-1]
    
    stmt_set_not_null = f"""ALTER TABLE {table_fqn} ALTER COLUMN {column} SET NOT NULL;
"""
    stmt_add_pk = f"""ALTER TABLE {table_fqn} 
ADD CONSTRAINT {table_name}_pk PRIMARY KEY({column});"""
    
    for stmt in [stmt_set_not_null, stmt_add_pk]:
        try:
            spark.sql(stmt)
        except Exception as e:
            print(f"Error executing sql: \n{stmt}")
            print(e)

In [0]:
foreign_keys = [
    ['silver_dev.geo.counties',
     'state_fips',
     'silver_dev.geo.states',
     'state_fips'
    ],
    ['silver_dev.healthcare.hospitals',
     'state_fips',
     'silver_dev.geo.states',
     'state_fips'
    ],

]

## Manually added foreign keys

In [0]:
%sql

/*
NOT ENFORCED RELY tells Databricks the constraint exists and is satisfied in your data, allowing the optimizer to use it
*/

ALTER TABLE silver_dev.geo.counties 
ADD CONSTRAINT counties_states_fk 
FOREIGN KEY (state_fips) REFERENCES silver_dev.geo.states(state_fips)
NOT ENFORCED RELY;

ALTER TABLE silver_dev.healthcare.hospitals 
ADD CONSTRAINT hospitals_counties_fk 
FOREIGN KEY (county_fips) REFERENCES silver_dev.geo.counties(county_fips)
NOT ENFORCED RELY;

ALTER TABLE silver_dev.demographics.county_population 
ADD CONSTRAINT county_population_counties_fk 
FOREIGN KEY (county_fips) REFERENCES silver_dev.geo.counties(county_fips)
NOT ENFORCED RELY;

ALTER TABLE silver_dev.demographics.county_poverty 
ADD CONSTRAINT county_poverty_counties_fk 
FOREIGN KEY (county_fips) REFERENCES silver_dev.geo.counties(county_fips)
NOT ENFORCED RELY;

ALTER TABLE silver_dev.demographics.ruca_census_tract 
ADD CONSTRAINT ruca_census_tract_counties_fk 
FOREIGN KEY (county_fips_23) REFERENCES silver_dev.geo.counties(county_fips)
NOT ENFORCED RELY;

ALTER TABLE silver_dev.demographics.ruca_census_tract 
ADD CONSTRAINT ruca_code_fk 
FOREIGN KEY (primary_ruca) REFERENCES silver_dev.demographics.ruca_codes(ruca_code)
NOT ENFORCED RELY;

ALTER TABLE gold_dev.demographics.ruca_county 
ADD CONSTRAINT ruca_county_fk 
FOREIGN KEY (county_fips) REFERENCES silver_dev.geo.counties(county_fips)
NOT ENFORCED RELY;

ALTER TABLE gold_dev.healthcare.county_hospital_summary 
ADD CONSTRAINT county_hospital_fk 
FOREIGN KEY (county_fips) REFERENCES silver_dev.geo.counties(county_fips)
NOT ENFORCED RELY;
